> 在线讲义：[第 48 章 · 推测解码：一次验证，确定好几个 token](https://wikieden.github.io/ai-infra-from-constraints/lessons/48-speculative.html) · 本书《从约束推导 AI Infra》

# 第 48 章 · 推测解码：一次验证，确定好几个 token

配套讲义：仓库里 `ai-infra/notes/lessons/48-speculative.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



接受概率 min(1, p/q)、每轮期望输出、每 token 耗时、查询时间的交点、草稿来源与草稿长度取自 AII 第 8 章 §8.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import random

**函数 `accept_prob`**：小工具。

In [ ]:
def accept_prob(p: float, q: float) -> float:
    return min(1.0, p / q)

**函数 `speculative_sample`**：从 q 采草稿，按 min(1, p/q) 接受，否则从 [p − q]+ 归一化后重采。

In [ ]:
def speculative_sample(p: dict[str, float], q: dict[str, float], rng: random.Random) -> str:
    """从 q 采草稿，按 min(1, p/q) 接受，否则从 [p − q]+ 归一化后重采。"""
    x = rng.choices(list(q), weights=list(q.values()))[0]
    if rng.random() < accept_prob(p[x], q[x]):
        return x
    resid = {k: max(p[k] - q.get(k, 0), 0) for k in p}
    return rng.choices(list(resid), weights=list(resid.values()))[0]

**函数 `expected_tokens`**：E[N] = 1 + Σ_j Π_{i≤j} a_i。

In [ ]:
def expected_tokens(a: float | list[float], m: int = 4) -> float:
    """E[N] = 1 + Σ_j Π_{i≤j} a_i。"""
    acc = [a] * m if isinstance(a, float) else a
    total, prod = 1.0, 1.0
    for ai in acc[:m]:
        prod *= ai
        total += prod
    return total

**函数 `per_token_ms`**：小工具。

In [ ]:
def per_token_ms(round_ms: float, n: float) -> float:
    return round_ms / n

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：保持目标分布（p(A) = 1/4，p(B) = 3/4）")
rng = random.Random(0)
p = {"A": 0.25, "B": 0.75}
for name, q in (("草稿恒为 A", {"A": 1.0, "B": 0.0}), ("草稿恒为 B", {"A": 0.0, "B": 1.0})):
    s = [speculative_sample(p, {k: v for k, v in q.items() if v > 0}, rng) for _ in range(20000)]
    print(f"  {name}：接受概率 {accept_prob(p[max(q, key=q.get)], 1.0):.2f}，采样得 P(A) ≈ {s.count('A') / len(s):.3f}")

In [ ]:
print("\n第二步：每轮输出与每 token 耗时（验证 26.26 ms + 查询 0.1 ms）")
for name, a in (("AAAA", 0.25), ("BBBB", 0.75)):
    n = expected_tokens(a)
    print(f"  {name}：E[N] = {n:.2f}，每 token {per_token_ms(26.36, n):.2f} ms；查询上限 {n * 26.26 - 26.26:.1f} ms")
print(f"  两轮 1.5 ms 各出 1、5 个：{3 / 6:.1f} ms/token（不是 {(1.5 + 0.3) / 2:.1f}）")
save = 26.26 - per_token_ms(26.36, expected_tokens(0.75))
print(f"  建索引 2 s：{math.ceil(2000 / save)} 个 token 回本；256 token {256 * 26.26 / 1e3:.2f} → {2 + 256 * per_token_ms(26.36, expected_tokens(0.75)) / 1e3:.2f} s")

In [ ]:
print("\n第三步：草稿来源（DFlash 2.10 GB，588 GB/s）")
fwd = 2.10e9 / 588e9 * 1e3
print(f"  一次前向 {fwd:.2f} ms；串行 4 次 {4 * fwd:.2f} ms")
print(f"  每轮 3 个 token：串行 {(4 * fwd + 26.26) / 3:.1f}，并行 {(fwd + 26.26) / 3:.1f} ms；并行只出 2 个 {(fwd + 26.26) / 2:.1f} ms")

In [ ]:
print("\n第四步：草稿长度（a = 3/4，T = 26.3 + 3.6m）")
for m in (1, 2, 4, 8):
    n = expected_tokens(0.75, m)
    print(f"  m = {m}：E[N] = {n:.2f}，{(26.3 + 3.6 * m) / n:.1f} ms/token")

## 3. 代码里才看得到的

1. 测试用三种草稿分布（全 A、全 B、各半）各采 4 万次，都和目标的 0.25 相差不到 0.01。
2. 256 个 token 的总时间代码得 4.21 s，原文 4.22 s：每 token 8.64 的末位舍入。
3. 并行草稿那行只换了草稿生成时间，验证的 26.26 ms 不变。

## 4. 自己改一改

- 让 `expected_tokens` 接受逐位置不同的 $a_i$（比如 0.9、0.8、0.6、0.4），找最优长度。
- 给每 token 耗时加"补齐到 8 行"的形状规则，复现 4 vs 5 的比较。
- 模拟 batch 16 时验证一轮变成 27.35 ms、草稿占显存 3 GiB，算还值不值。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 学生写草稿老师一次改，错处后面全划掉，至少定一个；
> 按 p 比 q 接受，拒了从差额重抽，分布不走样；
> 一趟车多带几件，平均要总时间除总件数；
> 猜得越远越难活，多猜一个要比原来便宜才猜。